In [1]:
# Cell 1: Title and overview

# SkillGap Analyzer - Full ML Notebook

"""
This notebook trains a job-role prediction model (TF-IDF + LogisticRegression), extracts role->required skills,
and provides a `gap_analysis()` utility that compares a candidate's skills to role requirements.

Cells:
1. Imports and config
2. Helpers: cleaning/tokenization
3. Load dataset
4. Prepare labels and safe split
5. Train TF-IDF pipeline
6. Save artifacts
7. predict_topk helper + examples
8. Build role->required skills mapping and save
9. gap_analysis utility + examples
10. Evaluate gap coverage on a sample
"""


"\nThis notebook trains a job-role prediction model (TF-IDF + LogisticRegression), extracts role->required skills,\nand provides a `gap_analysis()` utility that compares a candidate's skills to role requirements.\n\nCells:\n1. Imports and config\n2. Helpers: cleaning/tokenization\n3. Load dataset\n4. Prepare labels and safe split\n5. Train TF-IDF pipeline\n6. Save artifacts\n7. predict_topk helper + examples\n8. Build role->required skills mapping and save\n9. gap_analysis utility + examples\n10. Evaluate gap coverage on a sample\n"

In [2]:
# Cell 2: Environment and imports
import json
import logging
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s - %(message)s")
LOG = logging.getLogger('skillgap_notebook')

RANDOM_STATE = 42
ROOT = Path('..').resolve()  # notebook in ml-service/notebooks/
DATA_CSV = ROOT / 'dataset' / 'candidate_job_role_dataset.csv'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG.info('Data CSV: %s', DATA_CSV)
LOG.info('Artifacts dir: %s', ARTIFACT_DIR)


2026-04-03 22:36:23,929 INFO - Data CSV: F:\devx\projectdev\sgp\skilllensai\ml-service\dataset\candidate_job_role_dataset.csv
2026-04-03 22:36:23,930 INFO - Artifacts dir: F:\devx\projectdev\sgp\skilllensai\ml-service\artifacts


In [3]:
# Cell 3: Cleaning helpers
import re

def clean_skills(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = s.strip().lower()
    s = re.sub(r'\(.*?\)', ' ', s)
    s = re.sub(r'[^a-z0-9,+\s]', ' ', s)
    s = re.sub(r'[;|/]+', ',', s)
    s = re.sub(r'\s*,\s*', ',', s)
    toks = [t.strip() for t in s.split(',') if t.strip()]
    seen = set(); uniq = []
    for t in toks:
        if t not in seen:
            seen.add(t); uniq.append(t)
    return ' '.join(uniq)


def tokens_from_text(s: str) -> List[str]:
    return [t for t in clean_skills(s).split() if t]

# quick sanity
print(clean_skills('Python, SQL; Django (web) / Flask'))


python sql  django     flask


In [4]:
# Cell 4: Load dataset
assert DATA_CSV.exists(), f"Dataset not found: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)
assert 'skills' in df.columns and 'job_role' in df.columns, 'CSV must contain skills and job_role'
df = df[['skills', 'job_role']].dropna().copy()
df['skills_clean'] = df['skills'].apply(clean_skills)
LOG.info('Loaded %d rows', len(df))
df.head(5)


2026-04-03 22:36:48,663 INFO - Loaded 1000 rows


,skills,job_role,skills_clean
0,"Python, SQL, TensorFlow, Machine Learning, Com...",Data Scientist,python sql tensorflow machine learning communi...
1,"HTML, CSS, JavaScript, React, Teamwork",Frontend Developer,html css javascript react teamwork
2,"Java, Spring, SQL, REST APIs, Problem Solving",Backend Developer,java spring sql rest apis problem solving
3,"Figma, Adobe XD, UI/UX Design, Creativity",Designer,figma adobe xd ui ux design creativity
4,"Python, Django, JavaScript, React, Agile",Full Stack Python Developer,python django javascript react agile


In [5]:
# Cell 5: Prepare labels and safe train/test split
le = LabelEncoder()
y = le.fit_transform(df['job_role'].astype(str).values)
X = df['skills_clean'].astype(str).tolist()

counts = Counter(y)
min_count = min(counts.values()) if counts else 0
if min_count < 2:
    LOG.warning('Least populated class has %d samples — using non-stratified split', min_count)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

LOG.info('Train size: %d, Test size: %d', len(X_train), len(X_test))


2026-04-03 22:36:53,705 WARNING - Least populated class has 1 samples — using non-stratified split
2026-04-03 22:36:53,711 INFO - Train size: 800, Test size: 200


In [6]:
# Cell 6: Train TF-IDF + LogisticRegression pipeline
vec = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
clf = LogisticRegression(max_iter=2000, solver='saga', multi_class='multinomial', class_weight='balanced', random_state=RANDOM_STATE)
from sklearn.pipeline import Pipeline
pipeline = Pipeline([('tfidf', vec), ('clf', clf)])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
LOG.info('Test Accuracy: %.4f', acc)
labels = list(range(len(le.classes_)))
print(classification_report(y_test, y_pred, labels=labels, target_names=list(le.classes_), zero_division=0))


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2026-04-03 22:37:02,479 INFO - Test Accuracy: 1.0000


                                precision    recall  f1-score   support

                          AIML       1.00      1.00      1.00         7
             Backend Developer       1.00      1.00      1.00         7
          Blockchain Developer       1.00      1.00      1.00         7
                  C# Developer       1.00      1.00      1.00        11
        Cybersecurity Engineer       1.00      1.00      1.00        13
                  Data Analyst       1.00      1.00      1.00        11
                Data Scientist       1.00      1.00      1.00         9
                      Designer       1.00      1.00      1.00        12
               DevOps Engineer       1.00      1.00      1.00         8
                       Finance       0.00      0.00      0.00         0
            Frontend Developer       1.00      1.00      1.00         8
     Full Stack Java Developer       1.00      1.00      1.00         5
   Full Stack Python Developer       1.00      1.00      1.00  

In [7]:
# Cell 7: Save artifacts (pipeline, label encoder, mapping)
joblib.dump(pipeline, ARTIFACT_DIR / 'jobrole_pipeline.joblib')
joblib.dump(le, ARTIFACT_DIR / 'label_encoder.joblib')
meta = {'mode': 'tfidf', 'random_state': RANDOM_STATE}
with open(ARTIFACT_DIR / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
LOG.info('Saved pipeline, label encoder, and meta to %s', ARTIFACT_DIR)


2026-04-03 22:37:20,163 INFO - Saved pipeline, label encoder, and meta to F:\devx\projectdev\sgp\skilllensai\ml-service\artifacts


In [8]:
# Cell 8: predict_topk helper and examples
def predict_topk(pipeline, label_encoder, skills: str, k: int = 5):
    x = clean_skills(skills)
    probs = pipeline.predict_proba([x])[0]
    idx = np.argsort(probs)[::-1][:k]
    roles = [label_encoder.classes_[int(i)] for i in idx]
    return roles, [float(probs[int(i)]) for i in idx]

examples = [
    'python, pandas, scikit-learn, sql',
    'react, javascript, css, html',
    'aws, docker, kubernetes, terraform',
]
for ex in examples:
    r, p = predict_topk(pipeline, le, ex, k=5)
    print('Skills:', ex)
    print('Predicted roles (top5):', r)
    print('Probs:', p)
    print('---')


Skills: python, pandas, scikit-learn, sql
Predicted roles (top5): ['Data Analyst', 'Finance', 'Data Scientist', 'Full Stack Python Developer', 'Backend Developer']
Probs: [0.48216859690310965, 0.06639243082527606, 0.055054465063851456, 0.04284240508122378, 0.03465062477686695]
---
Skills: react, javascript, css, html
Predicted roles (top5): ['Frontend Developer', 'Finance', 'Web Developer', 'Full Stack Java Developer', 'Full Stack Python Developer']
Probs: [0.6445244646418166, 0.04992601763379221, 0.04907890410439905, 0.01989460489378071, 0.01967748302451518]
---
Skills: aws, docker, kubernetes, terraform
Predicted roles (top5): ['Kubernetes Operations Engineer', 'DevOps Engineer', 'Finance', 'Full Stack Java Developer', 'Mobile Developer']
Probs: [0.32183815507739616, 0.21493429115403728, 0.08489656447847219, 0.033734917748137735, 0.026254001050076168]
---


In [9]:
# Cell 9: Build role -> required skills (top-K) and save
TOP_K = 10
role_skill_counts = defaultdict(Counter)
for _, r in df.iterrows():
    role = r['job_role']
    toks = tokens_from_text(r['skills'])
    role_skill_counts[role].update(toks)

role_required = {role: [s for s, _ in cnt.most_common(TOP_K)] for role, cnt in role_skill_counts.items()}
# save artifacts
pd.DataFrame([{'role': r, 'required_skills': ','.join(ss)} for r, ss in role_required.items()]).to_csv(ARTIFACT_DIR / 'role_required_skills_top10.csv', index=False)
with open(ARTIFACT_DIR / 'role_required_skills_top10.json', 'w', encoding='utf-8') as f:
    json.dump(role_required, f, indent=2)
LOG.info('Wrote role_required artifacts to %s', ARTIFACT_DIR)


2026-04-03 22:37:35,459 INFO - Wrote role_required artifacts to F:\devx\projectdev\sgp\skilllensai\ml-service\artifacts


In [10]:
# Cell 10: gap_analysis utility + examples
def gap_analysis(candidate_skills: str, role: str, role_map: Dict[str, List[str]] = role_required) -> Dict:
    cand = tokens_from_text(candidate_skills)
    cand_set = set([c.lower() for c in cand if c])
    required = role_map.get(role, [])
    required_set = set([r.lower() for r in required])
    present = sorted(list(required_set & cand_set))
    missing = sorted(list(required_set - cand_set))
    coverage = (len(present) / len(required_set) * 100) if required_set else 0.0
    return {
        'role': role,
        'required_skills': required,
        'present_skills': present,
        'missing_skills': missing,
        'coverage_pct': round(coverage, 2),
    }

# example with a predicted role
sample = 'python, flask, sql, git'
pred_role = predict_topk(pipeline, le, sample, k=1)[0][0]
print('Predicted role:', pred_role)
print(gap_analysis(sample, pred_role))


Predicted role: Full Stack Python Developer
{'role': 'Full Stack Python Developer', 'required_skills': ['python', 'javascript', 'postgresql', 'django', 'flask', 'sql', 'react', 'agile'], 'present_skills': ['flask', 'python', 'sql'], 'missing_skills': ['agile', 'django', 'javascript', 'postgresql', 'react'], 'coverage_pct': 37.5}


In [11]:
# Cell 11: Evaluate gap coverage across a sample of the dataset
def evaluate_coverage(df_sample: pd.DataFrame, role_map: Dict[str, List[str]]):
    rows = []
    for _, r in df_sample.iterrows():
        out = gap_analysis(r['skills'], r['job_role'], role_map)
        rows.append({'role': out['role'], 'coverage_pct': out['coverage_pct'], 'missing_count': len(out['missing_skills'])})
    return pd.DataFrame(rows)

cov_df = evaluate_coverage(df.sample(frac=0.2, random_state=RANDOM_STATE), role_required)
cov_df.describe()


,coverage_pct,missing_count
count,200.000000,200.0000
mean,55.187900,3.8750
std,12.602915,1.4629
min,30.000000,1.0000
25%,50.000000,3.0000
50%,50.000000,4.0000
75%,60.000000,5.0000
max,85.710000,7.0000


In [ ]:
# Cell 12: Notes

# Next steps and production notes
# - Use the `role_required_skills_top10.json` in the inference API for real-time gap analysis (see `ml-service/app_gap.py`).
# - Improve required-skill extraction by computing per-role TF-IDF or using domain expert lists.
# - Add unit tests that assert behavior of `clean_skills()`, `predict_topk()` and `gap_analysis()`.
